In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from datetime import datetime, timezone
import os
from load_dotenv import load_dotenv
from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import lit
from pyspark.sql.window import Window
import re
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, DoubleType
load_dotenv()

aws_region = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION") or "us-east-1"

In [2]:
catalog_name = "glue_catalog"
database_name = "silver"
database_name_src = "bronze"
table_name = "jobs"
bucket_name = "amzn-s3-job-prj"
warehouse_path = f"s3://{bucket_name}/"

spark = SparkSession.builder \
    .appName("iceberg-s3-aws") \
    .master("local[*]") \
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
        "org.apache.hadoop:hadoop-aws:3.3.4"
    ) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config(f"spark.sql.catalog.{catalog_name}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{catalog_name}.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog") \
    .config(f"spark.sql.catalog.{catalog_name}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config(f"spark.sql.catalog.{catalog_name}.warehouse", warehouse_path) \
    .config(f"spark.sql.catalog.{catalog_name}.client.region", aws_region) \
    .config("spark.sql.defaultCatalog", catalog_name) \
    .config("spark.hadoop.fs.s3a.region", aws_region) \
    .config("spark.hadoop.fs.s3a.endpoint", f"s3.{aws_region}.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.path.style.access", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.EnvironmentVariableCredentialsProvider") \
    .config("spark.driver.extraJavaOptions", f"-Daws.region={aws_region}") \
    .config("spark.executor.extraJavaOptions", f"-Daws.region={aws_region}") \
    .getOrCreate()


your 131072x1 screen size is bogus. expect trouble
26/05/15 17:48:43 WARN Utils: Your hostname, kien resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/15 17:48:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/k/job_ete/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ntk04/.ivy2/cache
The jars for the packages stored in: /home/ntk04/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-eeee1aaf-acc1-4374-ac47-ad3500f8b710;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.5.0 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 386ms :: artifacts dl 8ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-aws-bundle;1.5.0 from centra

In [3]:
now = datetime.now(timezone.utc)
today = now.strftime("%Y-%m-%d")

In [4]:
df = (
    spark.read
        .format("iceberg")
        .load(f"{catalog_name}.{database_name_src}.{table_name}")
        .filter(col("dt") == today)
)

In [5]:
df = df.filter(col("job_id").isNotNull())
df = df.dropna(subset=["company", "title"])
df = df.withColumn(
    "salary",
    when(
        col("salary").isNull() | (trim(col("salary")) == ""),
        "Thoả thuận"
    ).otherwise(col("salary"))
)

In [6]:
def parse_salary_numbers(text):
    if text is None:
        return []

    raw = str(text).lower().strip()

    is_usd = bool(re.search(r"(\$|usd|dollar|đô)", raw))

    if is_usd:
        # USD: 3,500 -> 3500
        raw = raw.replace(",", "")
    else:
        # VND triệu: 11,5 triệu -> 11.5
        raw = raw.replace(",", ".")

    nums = re.findall(r"\d+(?:\.\d+)?", raw)
    return [float(x) for x in nums]


parse_salary_udf = F.udf(parse_salary_numbers, ArrayType(DoubleType()))

In [7]:
def clean_salary(df, col_name="salary", usd_to_vnd=28000):
    df = df.withColumn("salary_raw", F.col(col_name))

    df = df.withColumn(
        "salary_clean",
        F.lower(F.trim(F.col(col_name)))
    )

    # Detect currency
    df = df.withColumn(
        "salary_currency",
        F.when(
            F.col("salary_clean").rlike(r"(\$|usd|dollar|đô)"),
            F.lit("USD")
        ).otherwise(F.lit("VND"))
    )

    # Detect salary type
    df = df.withColumn(
        "salary_type",
        F.when(
            F.col("salary_clean").rlike(r"(thoả thuận|thỏa thuận|thoa thuan|cạnh tranh|canh tranh|negotiable)"),
            F.lit("negotiable")
        )
        .when(
            F.col("salary_clean").rlike(r"(tới|đến|toi|den|up to|max)"),
            F.lit("up_to")
        )
        .when(
            F.col("salary_clean").rlike(r"(từ|tu|from|min)"),
            F.lit("from")
        )
        .when(
            F.col("salary_clean").rlike(r"\d+"),
            F.lit("range")
        )
        .otherwise(F.lit("unknown"))
    )

    # Detect unit
    df = df.withColumn(
        "salary_unit",
        F.when(F.col("salary_type") == "negotiable", F.lit(None))
        .when(F.col("salary_currency") == "USD", F.lit("usd"))
        .when(F.col("salary_clean").rlike(r"(triệu|trieu)"), F.lit("million_vnd"))
        .when(F.col("salary_clean").rlike(r"(vnd|vnđ|đồng|dong)"), F.lit("vnd"))
        .otherwise(F.lit("million_vnd"))
    )

    # Extract numbers
    df = df.withColumn(
        "numbers",
        parse_salary_udf(F.col(col_name))
    )

    # Raw min
    df = df.withColumn(
        "salary_min_raw",
        F.when(
            F.col("salary_type").isin("from", "range"),
            F.col("numbers").getItem(0)
        )
    )

    # Raw max
    df = df.withColumn(
        "salary_max_raw",
        F.when(
            F.col("salary_type") == "up_to",
            F.col("numbers").getItem(0)
        )
        .when(
            (F.col("salary_type") == "range") & (F.size(F.col("numbers")) > 1),
            F.col("numbers").getItem(1)
        )
    )

    # Convert min to VND
    df = df.withColumn(
        "salary_min_vnd",
        F.when(F.col("salary_min_raw").isNull(), F.lit(None).cast(DoubleType()))
        .when(F.col("salary_currency") == "USD", F.col("salary_min_raw") * F.lit(usd_to_vnd))
        .when(F.col("salary_unit") == "million_vnd", F.col("salary_min_raw") * F.lit(1000000))
        .otherwise(F.col("salary_min_raw"))
    )

    # Convert max to VND
    df = df.withColumn(
        "salary_max_vnd",
        F.when(F.col("salary_max_raw").isNull(), F.lit(None).cast(DoubleType()))
        .when(F.col("salary_currency") == "USD", F.col("salary_max_raw") * F.lit(usd_to_vnd))
        .when(F.col("salary_unit") == "million_vnd", F.col("salary_max_raw") * F.lit(1000000))
        .otherwise(F.col("salary_max_raw"))
    )

    # Average salary
    df = df.withColumn(
        "salary_avg_vnd",
        F.when(
            F.col("salary_min_vnd").isNotNull() & F.col("salary_max_vnd").isNotNull(),
            (F.col("salary_min_vnd") + F.col("salary_max_vnd")) / 2
        )
        .when(F.col("salary_min_vnd").isNotNull(), F.col("salary_min_vnd"))
        .when(F.col("salary_max_vnd").isNotNull(), F.col("salary_max_vnd"))
    )

    # Parse flag
    df = df.withColumn(
        "is_salary_parsed",
        F.when(F.col("salary_type") == "negotiable", F.lit(True))
        .when(F.size(F.col("numbers")) > 0, F.lit(True))
        .otherwise(F.lit(False))
    )

    return df.drop("salary_clean")\
            .drop("numbers")\
            .drop("salary_min_raw")\
            .drop("salary_max_raw")\
            .drop("salary_unit")


In [8]:
df = clean_salary(df, "salary", usd_to_vnd=28000)

In [9]:
# clean text columns for silver layer
def clean_text(df, col_name):
    return df.withColumn(
        col_name,
        trim(
            regexp_replace(
                regexp_replace(regexp_replace(col(col_name), r"<[^>]+>", " "), r"[\r\n\t]+", " "),
                r"\s{2,}", " "
            )
        )
    )


def clean_skills(df, col_name="skills"):
    return df.withColumn(
        col_name,
        trim(
            regexp_replace(
                regexp_replace(lower(col(col_name)), r"[\r\n\t]+", " "),
                r"[;|/\\|]+", ","
            )
        )
    )

# normalize text fields
text_columns = ["job_description", "title", "location"]
for c in text_columns:
    if c in df.columns:
        df = clean_text(df, c)

if "skills" in df.columns:
    df = clean_skills(df, "skills")

# normalize crawl_time and dedupe latest row per business key
if "crawl_time" in df.columns:
    df = df.withColumn("crawl_time", to_timestamp(col("crawl_time")))

if "job_id" in df.columns:
    key_columns = ["job_id"]
    merge_condition = "tgt.job_id = src.job_id"
else:
    key_columns = ["company", "title", "location"]
    merge_condition = "tgt.company = src.company AND tgt.title = src.title AND tgt.location = src.location"

if "crawl_time" in df.columns:
    df = df.withColumn(
        "row_num",
        row_number().over(Window.partitionBy(*key_columns).orderBy(col("crawl_time").desc_nulls_last()))
    ).filter(col("row_num") == 1).drop("row_num")
else:
    df = df.dropDuplicates(key_columns)

# write to Delta silver table with merge for upserts



In [10]:
df.show(5, truncate=False)  

+--------------------------------------------------------------------------------------+--------------------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
df = (
    df
    .withColumnRenamed("experience", "experience_raw")
    .withColumn("experience_clean", F.lower(F.trim(F.col("experience_raw"))))
    .withColumn(
        "experience_type",
        F.when(F.col("experience_clean").rlike("không yêu cầu|khong yeu cau"), "no_required")
         .when(F.col("experience_clean").rlike("thoả thuận|thỏa thuận|thoa thuan"), "negotiable")
         .when(F.col("experience_clean").rlike(r"\d+"), "required")
         .otherwise("unknown")
    )
    .withColumn(
        "experience_years_min",
        F.when(F.col("experience_type") == "no_required", F.lit(0))
         .otherwise(F.regexp_extract(F.col("experience_clean"), r"(\d+)", 1).cast("int"))
    )
    .drop("experience_clean")
)

In [ ]:
df = (
    df
    .withColumn("company", F.trim(F.col("company_raw")))
    .withColumn("company", F.regexp_replace("company_clean", r"\s+", " "))
)

In [11]:
from pyspark.sql import functions as F

province_pattern = (
    r"Hà Nội|Hồ Chí Minh|Đà Nẵng|Hải Phòng|Cần Thơ|"
    r"Bình Dương|Đồng Nai|Bắc Ninh|Hưng Yên|Long An|"
    r"Thanh Hóa|Nghệ An|Khánh Hòa|Bà Rịa - Vũng Tàu"
)

# Clean location
df_clean = (
    df
    .withColumn("location_raw", F.col("location"))
    .withColumn("location_clean", F.trim(F.col("location")))
    .withColumn("location_clean", F.regexp_replace("location_clean", r"^\s*-\s*", ""))
    .withColumn("location_clean", F.regexp_replace("location_clean", r"\s+", " "))
)

# Đánh dấu điểm bắt đầu của location mới
# Ví dụ: "... cũ) - Hà Nội: Sunshine Center..."
# thành: "... cũ)|||Hà Nội: Sunshine Center..."
df_clean = df_clean.withColumn(
    "location_marked",
    F.regexp_replace(
        F.col("location_clean"),
        rf"\s*-\s*({province_pattern})\s*:",
        r"|||$1:"
    )
)

# Split thành array
df_clean = df_clean.withColumn(
    "location_items",
    F.split(F.col("location_marked"), r"\|\|\|")
)

# Tạo bảng job_locations
df_job_locations = (
    df_clean
    .select(
        "job_id",
        F.posexplode("location_items").alias("location_order", "location_raw_item")
    )
    .withColumn("location_raw_item", F.trim(F.col("location_raw_item")))
    .filter(F.col("location_raw_item") != "")
    .withColumn(
        "province_city",
        F.regexp_extract(
            F.col("location_raw_item"),
            rf"^({province_pattern})\s*:",
            1
        )
    )
    .withColumn(
        "location_detail",
        F.trim(
            F.regexp_replace(
                F.col("location_raw_item"),
                rf"^({province_pattern})\s*:\s*",
                ""
            )
        )
    )
)

In [12]:
df_job_locations.show(20, truncate=False)

+-------+--------------+-----------------------------------------------------------------------------------------------------------------------------+-------------+----------------------------------------------------------------------------------------------------------------+
|job_id |location_order|location_raw_item                                                                                                            |province_city|location_detail                                                                                                 |
+-------+--------------+-----------------------------------------------------------------------------------------------------------------------------+-------------+----------------------------------------------------------------------------------------------------------------+
|1832660|0             |Hồ Chí Minh: Số 7 Lô C2 Đường 659, Khu nhà ở Phước Long B, Phường Phước Long (Thành phố Thủ Đức cũ)                          |Hồ Chí Minh  |Số

In [13]:
df.show(5)

+--------------------+--------------------+-------------+--------------------+-------+--------------------+--------------------+-------------+--------------------+-----------+--------------------+--------------------+----------+-------------+---------------+-----------+--------------+--------------+--------------+----------------+
|             company|          crawl_time|   experience|     job_description| job_id|             job_url|            location|       salary|              skills|source_page|               title|      ingestion_time|        dt|   salary_raw|salary_currency|salary_type|salary_min_vnd|salary_max_vnd|salary_avg_vnd|is_salary_parsed|
+--------------------+--------------------+-------------+--------------------+-------+--------------------+--------------------+-------------+--------------------+-----------+--------------------+--------------------+----------+-------------+---------------+-----------+--------------+--------------+--------------+----------------+
|

In [14]:
df_location_summary = (
    df_job_locations
    .groupBy("job_id")
    .agg(
        F.first("province_city", ignorenulls=True).alias("primary_province_city"),
        F.count("*").alias("location_count")
    )
    .withColumn(
        "is_multi_location",
        F.col("location_count") > 1
    )
   
)

df_job_postings = (
    df_clean
    .drop("location_clean", "location_marked", "location_items")
    .join(df_location_summary, on="job_id", how="left")
    .drop("location_raw")\
    .drop("location")
)

In [15]:
df_job_postings.show(5)

+-------+--------------------+--------------------+-------------+--------------------+--------------------+-------------+--------------------+-----------+--------------------+--------------------+----------+-------------+---------------+-----------+--------------+--------------+--------------+----------------+---------------------+--------------+-----------------+
| job_id|             company|          crawl_time|   experience|     job_description|             job_url|       salary|              skills|source_page|               title|      ingestion_time|        dt|   salary_raw|salary_currency|salary_type|salary_min_vnd|salary_max_vnd|salary_avg_vnd|is_salary_parsed|primary_province_city|location_count|is_multi_location|
+-------+--------------------+--------------------+-------------+--------------------+--------------------+-------------+--------------------+-----------+--------------------+--------------------+----------+-------------+---------------+-----------+--------------+--

In [17]:

full_table_name_postings = f"{catalog_name}.{database_name}.{table_name}_postings"
full_table_name_locations = f"{catalog_name}.{database_name}.{table_name}_locations"
if not spark.catalog.tableExists(full_table_name_postings):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog_name}.{database_name}")
    df_job_postings.writeTo(full_table_name_postings).create()
    df_job_locations.writeTo(full_table_name_locations).create()
else:

    df_job_postings.createOrReplaceTempView("tmp_silver_job_postings")
    df_job_locations.createOrReplaceTempView("tmp_silver_job_locations")
    merge_sql = f"""
    MERGE INTO {full_table_name_postings} AS tgt
    USING tmp_silver_job_postings AS src
    ON tgt.job_id = src.job_id
    WHEN MATCHED AND src.crawl_time > tgt.crawl_time THEN
      UPDATE SET *
    WHEN NOT MATCHED THEN
      INSERT *
    """
    spark.sql(merge_sql)
    
    df_affected_jobs = df_job_postings.select("job_id").distinct()
    df_affected_jobs.createOrReplaceTempView("tmp_affected_jobs")
    
    spark.sql(f"""
    DELETE FROM {full_table_name_locations}
    WHERE job_id IN (
        SELECT job_id FROM tmp_affected_jobs
    )
    """)
    
    spark.sql(f"""
    INSERT INTO  {full_table_name_locations}    
    SELECT *
    FROM tmp_silver_job_locations
    """)

In [18]:
spark.table(full_table_name_postings).show(5, truncate=False)

+-------+--------------------------------------------------------------------------------------+--------------------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------